In [1]:
!pip install -q ultralytics pyyaml

In [2]:
import ultralytics
import torch

print("Ultralytics version:", ultralytics.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Chưa có GPU, hãy bật GPU trong Settings.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics version: 8.4.188
PyTorch version: 2.11.0+cu128
CUDA khả dụng: True
GPU: Tesla T4


In [3]:

from pathlib import Path
import yaml
from ultralytics import YOLO

# 1. Đường dẫn dataset
dataset_dir = Path(
    "/kaggle/input/datasets/trandailamk18hl/dataset-trainrailway/Dataset_all"
)

# 2. Đọc cấu hình dataset
with open(dataset_dir / "data.yaml", "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)

data["train"] = str(dataset_dir / "train" / "images")
data["val"] = str(dataset_dir / "valid" / "images")
data.pop("test", None)
data.pop("valid", None)

# Lưu YAML mới vào thư mục có quyền ghi
kaggle_yaml = Path("/kaggle/working/data.yaml")
with open(kaggle_yaml, "w", encoding="utf-8") as f:
    yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)

# 3. Khởi tạo YOLO26s pretrained
model = YOLO("yolo26s.pt")

# 4. Huấn luyện
results = model.train(
    data=str(kaggle_yaml),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    patience=20,
    project="/kaggle/working/runs",
    name="railway",
    exist_ok=True,
    plots=True,
    save=True,
    seed=42,
    close_mosaic=10
)

# 5. Đường dẫn model
print("\nĐÃ TRAIN XONG!")
print("Best model: /kaggle/working/runs/railway/weights/best.pt")
print("Last model: /kaggle/working/runs/railway/weights/last.pt")


Ultralytics 8.4.188 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=railway, nbs=64, nms=None, opset=None, optimize=False, optimizer

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/50      5.87G      1.125     0.8274   0.005005        141        640: 100% ━━━━━━━━━━━━ 133/133 4.0it/s 33.2s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 4.0it/s 4.3s0.3s
                   all        532       4586      0.869      0.838      0.882      0.658

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       3/50      5.88G      1.126     0.7628   0.005101        156        640: 100% ━━━━━━━━━━━━ 133/133 3.9it/s 34.0s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 4.1it/s 4.2s0.3s
                   all        532       4586      0.877      0.817      0.876      0.645

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       4/50      5.88G      1.124     0.7498    0.00502        236        640: 100% ━━━━━━━━━━━━ 133/133 4.0it/s 33.1s0.2s
                 Class     Images

In [6]:
import shutil

shutil.make_archive(
    '/kaggle/working/railway_output',
    'zip',
    '/kaggle/working/runs/railway'
)

'/kaggle/working/railway_output.zip'